# RobotMath2030 — Geometry of State (Ch.01–02)

SE(2) intuition + tiny pose graph SLAM.

Full chapter: [01_pose_is_not_vector](../chapters/01_pose_is_not_vector/)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    !git clone -q https://github.com/rsasaki0109/RobotMath2030.git 2>/dev/null || true
    %cd RobotMath2030
except ImportError:
    root = Path.cwd()
    if not (root / 'robotmath').exists() and (root.parent / 'robotmath').exists():
        %cd ..

!pip install -q -e .
%matplotlib inline

## Ch.01 — Euler interpolation failure (170° → −170°)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from robotmath.lie import so2

t_vals = np.linspace(0, 1, 50)
theta_a, theta_b = np.deg2rad(170), np.deg2rad(-170)
euclidean = [so2.wrap_angle(theta_a + t * (theta_b - theta_a)) for t in t_vals]
geodesic = [so2.wrap_angle(theta_a + t * so2.wrap_angle(theta_b - theta_a)) for t in t_vals]

plt.figure(figsize=(7, 4))
plt.plot(np.rad2deg(euclidean), 'C3--', label='linear on angles (wrong)')
plt.plot(np.rad2deg(geodesic), 'C0-', label='SO(2) geodesic')
plt.ylabel('heading [deg]')
plt.xlabel('interpolation parameter')
plt.title('Pose is not a vector')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Ch.02 — Lie vs Euclidean pose graph cost

In [ ]:
from miniworlds.pose_graph_world import square_loop_graph
from robotmath.lie import se2

graph, gt = square_loop_graph(seed=42)
g_lie = graph.copy()
g_euc = graph.copy()
opt_lie, cost_lie = g_lie.optimize(use_lie=True, max_iters=40)
opt_euc, cost_euc = g_euc.optimize(use_lie=False, max_iters=40)

def pose_error(poses, ref):
    return sum(float(se2.log(se2.between(T, G)) @ se2.log(se2.between(T, G))) for T, G in zip(poses, ref))

print(f'Lie final cost:       {cost_lie[-1]:.6f}')
print(f'Euclidean final cost: {cost_euc[-1]:.6f}')
print(f'Lie pose error:       {pose_error(opt_lie.poses, gt):.4f}')
print(f'Euclidean pose error: {pose_error(opt_euc.poses, gt):.4f}')